# Qwen3-1.7B Reasoning 两阶段微调（SFT → DPO）

**运行环境**：Google Colab（免费版 T4 / Pro A100 均可）

**流程**：安装依赖 → 挂载 Drive → 克隆项目 → SFT 训练 → DPO 训练 → 合并权重 → 推理验证

> ⚠️ 请按顺序逐个 Cell 运行，断线重连后从 **Cell 4（挂载 Drive）** 继续即可。

In [ ]:
# Cell 1：安装依赖（每次新 Session 都需要运行）
!pip install unsloth trl peft bitsandbytes transformers datasets accelerate pyyaml -q
print("✅ 依赖安装完成")

In [ ]:
# Cell 2：设置 HuggingFace Token
# 请先在 Colab 左侧【🔑 Secrets】面板中添加 HF_TOKEN，再运行此 Cell
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("✅ HuggingFace Token 已加载")

In [ ]:
# Cell 3：挂载 Google Drive（防止断线丢失训练结果）
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"
!mkdir -p "{DRIVE_DIR}/outputs"
print(f"✅ Drive 已挂载，输出目录：{DRIVE_DIR}/outputs")

In [ ]:
# Cell 4：克隆项目代码
import os

PROJECT_DIR = "/content/6000Q-QwenMiniReason"

if not os.path.exists(PROJECT_DIR):
    !git clone https://github.com/yukiiii0730/6000Q-QwenMiniReason.git {PROJECT_DIR}
else:
    print("项目已存在，跳过克隆")

%cd {PROJECT_DIR}
print("✅ 当前目录:", os.getcwd())

In [ ]:
# Cell 5：检查 GPU 并自动适配配置
import torch
import yaml

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vram_gb  = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else 0
print(f"GPU : {gpu_name}")
print(f"显存: {vram_gb} GB")

DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"

with open("config/sft_config.yaml", "r") as f:
    sft_cfg = yaml.safe_load(f)

sft_cfg["output_dir"] = f"{DRIVE_DIR}/outputs/sft"
if "T4" in gpu_name:
    sft_cfg["train"]["fp16"] = True
    sft_cfg["train"]["bf16"] = False
    print("⚠️  检测到 T4，已切换为 fp16")
if vram_gb < 20:
    for ds in sft_cfg.get("datasets", []):
        ds["max_samples"] = 10000
    print("⚠️  显存 < 20GB，max_samples 已降至 10000")

with open("config/sft_config.yaml", "w") as f:
    yaml.dump(sft_cfg, f, allow_unicode=True)

with open("config/dpo_config.yaml", "r") as f:
    dpo_cfg = yaml.safe_load(f)

dpo_cfg["output_dir"]        = f"{DRIVE_DIR}/outputs/dpo"
dpo_cfg["base_adapter_path"] = f"{DRIVE_DIR}/outputs/sft"
if "T4" in gpu_name:
    dpo_cfg["train"]["fp16"] = True
    dpo_cfg["train"]["bf16"] = False
if vram_gb < 20:
    dpo_cfg["dataset"]["max_samples"] = 10000

with open("config/dpo_config.yaml", "w") as f:
    yaml.dump(dpo_cfg, f, allow_unicode=True)

print("✅ 配置已更新完成")

In [ ]:
# Cell 6：第一阶段 SFT 训练
!python scripts/sft_train.py --config config/sft_config.yaml

In [ ]:
# Cell 7：验证 SFT 产物是否已保存到 Drive
import os
DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"
print("SFT 输出文件:", os.listdir(f"{DRIVE_DIR}/outputs/sft"))

In [ ]:
# Cell 8：第二阶段 DPO 训练
!python scripts/dpo_train.py --config config/dpo_config.yaml

In [ ]:
# Cell 9：合并 LoRA 权重到基础模型
DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"
!python scripts/merge_lora.py \
    --adapter_path "{DRIVE_DIR}/outputs/dpo" \
    --output_path  "{DRIVE_DIR}/outputs/merged"
print("✅ 权重合并完成")

In [ ]:
# Cell 10：推理验证——对比基础模型 vs 微调模型
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

DRIVE_DIR  = "/content/drive/MyDrive/Qwen-Reasoning"
BASE_MODEL = "Qwen/Qwen3-1.7B"
FT_MODEL   = f"{DRIVE_DIR}/outputs/merged"
QUESTION   = "小明有12个苹果，给了小红3个，又买了5个，现在有几个？请一步步推理后给出答案。"

def infer(model_path, prompt, max_new_tokens=512):
    tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_path, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True
    )
    inputs  = tok(prompt, return_tensors="pt").to(mdl.device)
    outputs = mdl.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(outputs[0], skip_special_tokens=True)

print("=" * 60)
print("【基础模型】")
print(infer(BASE_MODEL, QUESTION))
print()
print("=" * 60)
print("【微调模型（SFT + DPO）】")
print(infer(FT_MODEL, QUESTION))

In [ ]:
# Cell 11（可选）：GSM8K 自动评测
DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"
!python eval/gsm8k_eval.py \
    --model_path  "{DRIVE_DIR}/outputs/merged" \
    --max_samples 200 \
    --output      eval/gsm8k_result.json

import json
with open("eval/gsm8k_result.json") as f:
    result = json.load(f)
print(f"GSM8K Accuracy: {result['accuracy']:.2%}  ({result['correct']}/{result['total']})")

In [ ]:
# Cell 12（可选）：上传微调模型到 HuggingFace
from huggingface_hub import HfApi
import os

DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"
REPO_ID   = "yukiiii0730/Qwen3-1.7B-Math-Reasoning"  # ← 按需修改

api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(REPO_ID, exist_ok=True)
api.upload_folder(
    folder_path=f"{DRIVE_DIR}/outputs/merged",
    repo_id=REPO_ID,
    repo_type="model",
)
print(f"✅ 已上传至 https://huggingface.co/{REPO_ID}")